In [0]:
%%capture --no-stderr
%pip install --quiet -U databricks-langchain langchain_core langgraph
dbutils.library.restartPython()

In [0]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content=f"So you said you were researching ocean mammals?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right.",name="Lance"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best place to see Orcas in the US.", name="Lance"))

for m in messages:
    m.pretty_print()

In [0]:
from databricks_langchain import ChatDatabricks
llm = ChatDatabricks(model="agents-demo-gpt4o")
result = llm.invoke(messages)
type(result)

In [0]:
result

In [0]:
def multiply(a: int, b: int) -> int:
  """Multiply a and b.

    Args:
        a: first int
        b: second int
  """
  return a * b

llm_with_tools = llm.bind_tools([multiply])
llm_with_tools

In [0]:
llm_with_tools.invoke([HumanMessage(content="What is 2 * 3?", name="User")])

In [0]:
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

class State(TypedDict):
  messages: Annotated[list, add_messages]

In [0]:
from langgraph.graph import MessagesState

class State(MessagesState):
  pass

In [0]:
from langgraph.graph import StateGraph, START, END

def tool_calling(state: State) -> State:
  return {'messages': [llm_with_tools.invoke(state['messages'])]}

graph = StateGraph(State)


graph.add_node('tool_calling', tool_calling)

graph.add_edge(START, 'tool_calling')
graph.add_edge('tool_calling', END)

app = graph.compile()
app

In [0]:
messages = app.invoke({"messages": HumanMessage(content="Hello!")})
for m in messages['messages']:
    m.pretty_print()

In [0]:
messages = app.invoke({"messages": HumanMessage(content="what is 2 * 3!")})
for m in messages['messages']:
    m.pretty_print()